In [0]:
# ============================================
# GOLD LAYER PIPELINE
# ============================================

from pyspark.sql.functions import *

# ============================================
# DELETE OLD GOLD TABLES
# ============================================

dbutils.fs.rm(
    "/Volumes/workspace/default/lakehouse_raw_data/gold/top_customers",
    recurse=True
)

dbutils.fs.rm(
    "/Volumes/workspace/default/lakehouse_raw_data/gold/daily_revenue",
    recurse=True
)

dbutils.fs.rm(
    "/Volumes/workspace/default/lakehouse_raw_data/gold/revenue_by_country",
    recurse=True
)

# ============================================
# READ SILVER TABLES
# ============================================

orders_df = spark.read.format("delta").load(
    "/Volumes/workspace/default/lakehouse_raw_data/silver/orders_clean"
)

customers_df = spark.read.format("delta").load(
    "/Volumes/workspace/default/lakehouse_raw_data/silver/customers_clean"
)

# ============================================
# FILTER VALID SUCCESS ORDERS
# ============================================

success_orders_df = orders_df.filter(
    (col("status") == "SUCCESS") &
    (col("amount") > 0) &
    (col("customer_id").isNotNull())
)

# ============================================
# DAILY REVENUE
# ============================================

daily_revenue_df = success_orders_df.groupBy(
    "order_date"
).agg(
    round(sum("amount"), 2).alias("daily_revenue")
).orderBy(
    "order_date"
)

# ============================================
# TOP CUSTOMERS
# ============================================

top_customers_df = success_orders_df.join(
    customers_df,
    "customer_id",
    "inner"
).groupBy(
    "name"
).agg(
    round(sum("amount"), 2).alias("total_spent")
).orderBy(
    col("total_spent").desc()
)

# ============================================
# REVENUE BY COUNTRY
# ============================================

revenue_by_country_df = success_orders_df.join(
    customers_df,
    "customer_id",
    "inner"
).groupBy(
    "country"
).agg(
    round(sum("amount"), 2).alias("total_revenue")
).orderBy(
    col("total_revenue").desc()
)

# ============================================
# DISPLAY RESULTS
# ============================================

print("Top Customers")
display(top_customers_df.limit(20))

print("Daily Revenue")
display(daily_revenue_df.limit(20))

print("Revenue By Country")
display(revenue_by_country_df.limit(20))

# ============================================
# WRITE GOLD TABLES
# ============================================

top_customers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "/Volumes/workspace/default/lakehouse_raw_data/gold/top_customers"
    )

daily_revenue_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "/Volumes/workspace/default/lakehouse_raw_data/gold/daily_revenue"
    )

revenue_by_country_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(
        "/Volumes/workspace/default/lakehouse_raw_data/gold/revenue_by_country"
    )

# ============================================
# VALIDATION COUNTS
# ============================================

print("Top Customers Count:", top_customers_df.count())

print("Daily Revenue Count:", daily_revenue_df.count())

print("Revenue By Country Count:", revenue_by_country_df.count())

print("Gold Layer Created Successfully!")

Top Customers


name,total_spent
Jessica Smith,154490.46
David White,112474.3
Michael Smith,112452.81
Ashley Jackson,103501.46
Christopher Miller,102951.86
Ashley Richardson,100192.71
Julie Lee,96616.92
Jordan Jones,92498.89
Christopher Davis,92061.75
Michael Davis,89491.83


Daily Revenue


order_date,daily_revenue
2025-01-01,623581.08
2025-01-02,634099.11
2025-01-03,522342.88
2025-01-04,509208.23
2025-01-05,643519.35
2025-01-06,630601.67
2025-01-07,621982.97
2025-01-08,544573.66
2025-01-09,611879.66
2025-01-10,640430.86


Revenue By Country


country,total_revenue
USA,2.269067536E7
India,2.21704257E7
Brazil,2.160822529E7
Japan,2.148327079E7
Canada,2.102396981E7
UK,2.088020462E7
Germany,2.057515548E7
Australia,2.050444894E7
Singapore,2.032287347E7
France,2.026508294E7


Top Customers Count: 9370
Daily Revenue Count: 366
Revenue By Country Count: 10
Gold Layer Created Successfully!
